# 03_Modeling.ipynb

## **0. Giới thiệu**

Notebook này thực hiện:

- Phân tích dữ liệu sau khi đã preprocessing.

- Thực hiện các kiểm định thống kê cơ bản.

- Áp dụng mô hình Machine Learning đơn giản (Logistic Regression bằng NumPy).

- Đánh giá mô hình bằng accuracy, precision, recall, confusion matrix.

- Vẽ biểu đồ từ thư viện `src/visualization.py`.

## **1. Import & Load dữ liệu đã tiền xử lý**

In [ ]:
import numpy as np
import sys, os


# thêm đường dẫn để import src
project_root = os.path.abspath("..")
sys.path.append(project_root)


from src.models import logistic_regression, predict, confusion_matrix
from src.data_processing import train_test_split
from src.visualization import (
    plot_hist_numeric,
    plot_scatter,
    plot_corr_heatmap
)


# load dữ liệu sạch từ file 02
X = np.load("../data/processed/data_clean.npy", allow_pickle=True)
headers = np.load("../data/processed/headers_clean.npy", allow_pickle=True)

## **2. Xác định biến đầu vào & target**

In [ ]:
target_idx = headers.tolist().index("target")
y = X[:, target_idx].astype(float)


# loại bỏ target ra khỏi features
X_features = np.delete(X, target_idx, axis=1).astype(float)

print("Shape X:", X_features.shape)
print("Shape y:", y.shape)

Shape X: (19158, 14)
Shape y: (19158,)


## **3. Train-test split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, seed=0)

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)

## **4. Logistic Regression (tự cài bằng NumPy)**

In [ ]:
w = logistic_regression(X_train, y_train, lr=0.01, epochs=300)
print("Model weights shape:", w.shape)

## **5. Dự đoán & Đánh giá mô hình**

In [ ]:
train_pred = predict(X_train, w)
test_pred = predict(X_test, w)


train_acc = (train_pred == y_train).mean()
test_acc = (test_pred == y_test).mean()


print("Accuracy (train):", train_acc)
print("Accuracy (test): ", test_acc)

## **6. Confusion Matrix**

In [ ]:
cm = confusion_matrix(y_test, test_pred)
print("Confusion matrix:\n", cm)

## **7. Visualization trên dữ liệu sạch**

Histogram cho một số biến

In [ ]:
idx_train = headers.tolist().index("training_hours")
arr_train = X[:, idx_train].astype(float)
plot_hist_numeric(arr_train, title="Training Hours", xlabel="Training Hours")

Scatter đơn giản

In [ ]:
cdi = X[:, headers.tolist().index("city_development_index")].astype(float)
train = X[:, idx_train].astype(float)
plot_scatter(cdi, train, xlabel="City Dev Index", ylabel="Training Hours")

Heatmap tương quan

In [ ]:
numeric_cols = [h for h in headers if h not in ["target"]]
num_data = X_features.astype(float)
plot_corr_heatmap(num_data, numeric_cols)

## **8. Kiểm định giả thuyết thống kê (t-test)**

Giả thuyết:
- H0: `training_hours` của 2 nhóm (tìm việc vs không tìm việc) là như nhau
- H1: `training_hours` của nhóm tìm việc cao hơn

In [ ]:
train_hours = X[:, idx_train].astype(float)

train_0 = train_hours[y == 0]
train_1 = train_hours[y == 1]

mean0, mean1 = train_0.mean(), train_1.mean()
var0, var1 = train_0.var(), train_1.var()
n0, n1 = len(train_0), len(train_1)

t_stat = (mean1 - mean0) / np.sqrt(var1/n1 + var0/n0)
print("t-statistic:", t_stat)

## **9. Kết luận**

- Logistic Regression hoạt động ổn trên dữ liệu đã preprocessing
- Accuracy test OK (tuỳ dataset imbalance)
- Các feature chỉnh chuẩn (scaled, clean, encoded) → học tốt
- Visualization cho thấy mối quan hệ hợp lý
- T-test cho insight hành vi ứng viên